# Tarea: Analítica de Datos con PySpark
## Procesamiento de Grandes Bases de Datos

### Objetivo
Desarrollar un flujo completo de análisis de datos utilizando PySpark sobre un conjunto de datos en formato CSV, incluyendo los pasos de carga, limpieza, transformación, análisis y exportación de resultados.

In [1]:
# Importar librerías necesarias
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import pandas as pd

In [2]:
# Crear sesión de Spark
spark = SparkSession.builder \
    .appName("Analitica_Ventas_PySpark") \
    .getOrCreate()

print("Sesión de Spark creada exitosamente")

Sesión de Spark creada exitosamente


In [3]:
# Función para guardar CSV en la misma carpeta usando Pandas
def guardar_csv(dataframe, nombre_archivo, carpeta_salida='.'):
    """
    Guarda un DataFrame de PySpark como archivo CSV.
    Convierte a Pandas para evitar problemas de Hadoop en Windows.
    Por defecto guarda en la misma carpeta del notebook.
    
    Args:
        dataframe: DataFrame de PySpark a guardar
        nombre_archivo: Nombre del archivo CSV de salida (sin extensión)
        carpeta_salida: Carpeta donde guardar (default: '.' = carpeta actual)
    """
    # Convertir PySpark DataFrame a Pandas
    df_pandas = dataframe.toPandas()
    
    # Crear carpeta de salida si no existe (solo si no es directorio actual)
    if carpeta_salida != '.' and not os.path.exists(carpeta_salida):
        os.makedirs(carpeta_salida)
    
    # Guardar archivo directamente con Pandas
    output_path = os.path.join(carpeta_salida, f'{nombre_archivo}.csv')
    df_pandas.to_csv(output_path, index=False, encoding='utf-8-sig')
    
    print(f"✓ Guardado: {output_path}")

print("Función guardar_csv() definida")

Función guardar_csv() definida


## 1. Carga de Datos
Cargar el archivo ventas.csv utilizando spark.read.csv() con la opción header=True

In [5]:
# Cargar el archivo ventas.csv
df_ventas = spark.read.csv('ventas.csv', header=True, inferSchema=True)

print("Archivo ventas.csv cargado exitosamente")
print(f"Número total de registros: {df_ventas.count()}")

Archivo ventas.csv cargado exitosamente
Número total de registros: 6


## 2. Análisis Exploratorio de Datos
Mostrar el esquema y las primeras filas del DataFrame

In [6]:
# Mostrar esquema del DataFrame
print("Esquema del DataFrame:")
df_ventas.printSchema()

Esquema del DataFrame:
root
 |-- Producto: string (nullable = true)
 |-- Categoria: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Precio: integer (nullable = true)
 |-- Unidades_Vendidas: integer (nullable = true)



In [7]:
# Mostrar las primeras filas
print("Primeras filas del DataFrame:")
df_ventas.show()

Primeras filas del DataFrame:
+----------+-----------+------+------+-----------------+
|  Producto|  Categoria|Region|Precio|Unidades_Vendidas|
+----------+-----------+------+------+-----------------+
|    Laptop|Electrónica| Norte| 15000|               30|
|    Tablet|Electrónica|   Sur|  7000|               45|
|Smartphone|Electrónica|Centro|  9000|               60|
|   Monitor| Periférico| Norte|  4000|               25|
|   Teclado| Periférico|   Sur|   800|               80|
|     Mouse| Periférico|Centro|   600|               90|
+----------+-----------+------+------+-----------------+



In [8]:
# Mostrar estadísticas descriptivas
print("Estadísticas descriptivas:")
df_ventas.describe().show()

Estadísticas descriptivas:
+-------+--------+-----------+------+-----------------+------------------+
|summary|Producto|  Categoria|Region|           Precio| Unidades_Vendidas|
+-------+--------+-----------+------+-----------------+------------------+
|  count|       6|          6|     6|                6|                 6|
|   mean|    NULL|       NULL|  NULL|6066.666666666667|              55.0|
| stddev|    NULL|       NULL|  NULL| 5498.60588391882|26.457513110645905|
|    min|  Laptop|Electrónica|Centro|              600|                25|
|    max| Teclado| Periférico|   Sur|            15000|                90|
+-------+--------+-----------+------+-----------------+------------------+



In [9]:
# Contar registros por columna
print("Conteo de valores no nulos por columna:")
df_ventas.select([count(when(col(c).isNotNull(), c)).alias(c) for c in df_ventas.columns]).show()

Conteo de valores no nulos por columna:
+--------+---------+------+------+-----------------+
|Producto|Categoria|Region|Precio|Unidades_Vendidas|
+--------+---------+------+------+-----------------+
|       6|        6|     6|     6|                6|
+--------+---------+------+------+-----------------+



## 3. Limpieza de Datos
Eliminar registros nulos o inconsistentes utilizando dropna() o fillna()

In [10]:
# Verificar valores nulos
print("Verificación de valores nulos:")
df_ventas.select([count(when(col(c).isNull(), c)).alias(c) for c in df_ventas.columns]).show()

Verificación de valores nulos:
+--------+---------+------+------+-----------------+
|Producto|Categoria|Region|Precio|Unidades_Vendidas|
+--------+---------+------+------+-----------------+
|       0|        0|     0|     0|                0|
+--------+---------+------+------+-----------------+



In [11]:
# Eliminar filas con valores nulos (si existen)
df_ventas_limpio = df_ventas.dropna()

print(f"Registros después de limpieza: {df_ventas_limpio.count()}")
df_ventas_limpio.show()

Registros después de limpieza: 6
+----------+-----------+------+------+-----------------+
|  Producto|  Categoria|Region|Precio|Unidades_Vendidas|
+----------+-----------+------+------+-----------------+
|    Laptop|Electrónica| Norte| 15000|               30|
|    Tablet|Electrónica|   Sur|  7000|               45|
|Smartphone|Electrónica|Centro|  9000|               60|
|   Monitor| Periférico| Norte|  4000|               25|
|   Teclado| Periférico|   Sur|   800|               80|
|     Mouse| Periférico|Centro|   600|               90|
+----------+-----------+------+------+-----------------+



In [12]:
# Agregar columna calculada: Ingresos Totales (Precio * Unidades_Vendidas)
df_ventas_limpio = df_ventas_limpio.withColumn('Ingresos_Totales', 
                                                col('Precio') * col('Unidades_Vendidas'))

print("Columna 'Ingresos_Totales' agregada:")
df_ventas_limpio.show()

Columna 'Ingresos_Totales' agregada:
+----------+-----------+------+------+-----------------+----------------+
|  Producto|  Categoria|Region|Precio|Unidades_Vendidas|Ingresos_Totales|
+----------+-----------+------+------+-----------------+----------------+
|    Laptop|Electrónica| Norte| 15000|               30|          450000|
|    Tablet|Electrónica|   Sur|  7000|               45|          315000|
|Smartphone|Electrónica|Centro|  9000|               60|          540000|
|   Monitor| Periférico| Norte|  4000|               25|          100000|
|   Teclado| Periférico|   Sur|   800|               80|           64000|
|     Mouse| Periférico|Centro|   600|               90|           54000|
+----------+-----------+------+------+-----------------+----------------+



## 4. Operaciones Analíticas
Realizar agrupamientos, cálculos de promedios, máximos, mínimos y filtrados

In [13]:
# Análisis por Categoría
print("Análisis por Categoría:")
analisis_categoria = df_ventas_limpio.groupBy('Categoria').agg(
    count('Producto').alias('Total_Productos'),
    sum('Unidades_Vendidas').alias('Total_Unidades'),
    sum('Ingresos_Totales').alias('Ingresos_Totales'),
    avg('Precio').alias('Precio_Promedio'),
    max('Precio').alias('Precio_Maximo'),
    min('Precio').alias('Precio_Minimo')
).orderBy(col('Ingresos_Totales').desc())

analisis_categoria.show()

Análisis por Categoría:
+-----------+---------------+--------------+----------------+------------------+-------------+-------------+
|  Categoria|Total_Productos|Total_Unidades|Ingresos_Totales|   Precio_Promedio|Precio_Maximo|Precio_Minimo|
+-----------+---------------+--------------+----------------+------------------+-------------+-------------+
|Electrónica|              3|           135|         1305000|10333.333333333334|        15000|         7000|
| Periférico|              3|           195|          218000|            1800.0|         4000|          600|
+-----------+---------------+--------------+----------------+------------------+-------------+-------------+



In [14]:
# Análisis por Región
print("Análisis por Región:")
analisis_region = df_ventas_limpio.groupBy('Region').agg(
    count('Producto').alias('Total_Productos'),
    sum('Unidades_Vendidas').alias('Total_Unidades'),
    sum('Ingresos_Totales').alias('Ingresos_Totales'),
    avg('Precio').alias('Precio_Promedio')
).orderBy(col('Ingresos_Totales').desc())

analisis_region.show()

Análisis por Región:
+------+---------------+--------------+----------------+---------------+
|Region|Total_Productos|Total_Unidades|Ingresos_Totales|Precio_Promedio|
+------+---------------+--------------+----------------+---------------+
|Centro|              2|           150|          594000|         4800.0|
| Norte|              2|            55|          550000|         9500.0|
|   Sur|              2|           125|          379000|         3900.0|
+------+---------------+--------------+----------------+---------------+



In [15]:
# Análisis combinado: Categoría y Región
print("Análisis por Categoría y Región:")
analisis_combinado = df_ventas_limpio.groupBy('Categoria', 'Region').agg(
    sum('Unidades_Vendidas').alias('Total_Unidades'),
    sum('Ingresos_Totales').alias('Ingresos_Totales')
).orderBy(col('Ingresos_Totales').desc())

analisis_combinado.show()

Análisis por Categoría y Región:
+-----------+------+--------------+----------------+
|  Categoria|Region|Total_Unidades|Ingresos_Totales|
+-----------+------+--------------+----------------+
|Electrónica|Centro|            60|          540000|
|Electrónica| Norte|            30|          450000|
|Electrónica|   Sur|            45|          315000|
| Periférico| Norte|            25|          100000|
| Periférico|   Sur|            80|           64000|
| Periférico|Centro|            90|           54000|
+-----------+------+--------------+----------------+



In [16]:
# Filtrado: Productos con precio mayor a 5000
print("Productos con precio mayor a 5000:")
productos_premium = df_ventas_limpio.filter(col('Precio') > 5000)
productos_premium.show()

Productos con precio mayor a 5000:
+----------+-----------+------+------+-----------------+----------------+
|  Producto|  Categoria|Region|Precio|Unidades_Vendidas|Ingresos_Totales|
+----------+-----------+------+------+-----------------+----------------+
|    Laptop|Electrónica| Norte| 15000|               30|          450000|
|    Tablet|Electrónica|   Sur|  7000|               45|          315000|
|Smartphone|Electrónica|Centro|  9000|               60|          540000|
+----------+-----------+------+------+-----------------+----------------+



In [17]:
# Filtrado: Productos con más de 50 unidades vendidas
print("Productos con más de 50 unidades vendidas:")
productos_populares = df_ventas_limpio.filter(col('Unidades_Vendidas') > 50)
productos_populares.show()

Productos con más de 50 unidades vendidas:
+----------+-----------+------+------+-----------------+----------------+
|  Producto|  Categoria|Region|Precio|Unidades_Vendidas|Ingresos_Totales|
+----------+-----------+------+------+-----------------+----------------+
|Smartphone|Electrónica|Centro|  9000|               60|          540000|
|   Teclado| Periférico|   Sur|   800|               80|           64000|
|     Mouse| Periférico|Centro|   600|               90|           54000|
+----------+-----------+------+------+-----------------+----------------+



In [18]:
# Top 3 productos por ingresos totales
print("Top 3 productos por ingresos totales:")
top_productos = df_ventas_limpio.orderBy(col('Ingresos_Totales').desc()).limit(3)
top_productos.show()

Top 3 productos por ingresos totales:
+----------+-----------+------+------+-----------------+----------------+
|  Producto|  Categoria|Region|Precio|Unidades_Vendidas|Ingresos_Totales|
+----------+-----------+------+------+-----------------+----------------+
|Smartphone|Electrónica|Centro|  9000|               60|          540000|
|    Laptop|Electrónica| Norte| 15000|               30|          450000|
|    Tablet|Electrónica|   Sur|  7000|               45|          315000|
+----------+-----------+------+------+-----------------+----------------+



## 5. Consultas SQL
Crear vistas temporales y realizar consultas SQL sobre los datos

In [19]:
# Crear vista temporal
df_ventas_limpio.createOrReplaceTempView('ventas')

print("Vista temporal 'ventas' creada")

Vista temporal 'ventas' creada


In [20]:
# Consulta SQL: Total de ingresos por categoría
print("Consulta SQL - Total de ingresos por categoría:")
spark.sql("""
    SELECT Categoria,
           SUM(Ingresos_Totales) as Ingresos_Totales,
           COUNT(*) as Num_Productos
    FROM ventas
    GROUP BY Categoria
    ORDER BY Ingresos_Totales DESC
""").show()

Consulta SQL - Total de ingresos por categoría:
+-----------+----------------+-------------+
|  Categoria|Ingresos_Totales|Num_Productos|
+-----------+----------------+-------------+
|Electrónica|         1305000|            3|
| Periférico|          218000|            3|
+-----------+----------------+-------------+



In [21]:
# Consulta SQL: Producto más vendido por región
print("Consulta SQL - Producto más vendido por región:")
spark.sql("""
    SELECT Region,
           Producto,
           Unidades_Vendidas,
           Ingresos_Totales
    FROM ventas
    ORDER BY Region, Unidades_Vendidas DESC
""").show()

Consulta SQL - Producto más vendido por región:
+------+----------+-----------------+----------------+
|Region|  Producto|Unidades_Vendidas|Ingresos_Totales|
+------+----------+-----------------+----------------+
|Centro|     Mouse|               90|           54000|
|Centro|Smartphone|               60|          540000|
| Norte|    Laptop|               30|          450000|
| Norte|   Monitor|               25|          100000|
|   Sur|   Teclado|               80|           64000|
|   Sur|    Tablet|               45|          315000|
+------+----------+-----------------+----------------+



## 6. Exportación de Resultados
Guardar los resultados en archivos CSV en la misma carpeta del notebook

In [22]:
# Guardar DataFrame limpio con ingresos totales
print("Guardando resultados...\n")
guardar_csv(df_ventas_limpio, 'ventas_limpio')

Guardando resultados...

✓ Guardado: .\ventas_limpio.csv


In [23]:
# Guardar análisis por categoría
guardar_csv(analisis_categoria, 'analisis_categoria')

✓ Guardado: .\analisis_categoria.csv


In [24]:
# Guardar análisis por región
guardar_csv(analisis_region, 'analisis_region')

✓ Guardado: .\analisis_region.csv


In [25]:
# Guardar análisis combinado
guardar_csv(analisis_combinado, 'analisis_combinado')

✓ Guardado: .\analisis_combinado.csv


## 7. Resumen de Resultados

In [26]:
# Mostrar resumen de los análisis realizados
print("="*60)
print("RESUMEN DE ANÁLISIS DE VENTAS")
print("="*60)

print("\n1. Datos Cargados:")
print(f"   - Total de registros: {df_ventas_limpio.count()}")
print(f"   - Total de columnas: {len(df_ventas_limpio.columns)}")

print("\n2. Análisis por Categoría:")
analisis_categoria.show(truncate=False)

print("\n3. Análisis por Región:")
analisis_region.show(truncate=False)

print("\n4. Archivos CSV generados en la carpeta actual:")
archivos_csv = [f for f in os.listdir('.') if f.endswith('.csv')]
for archivo in sorted(archivos_csv):
    print(f"   - {archivo}")

print("\n" + "="*60)
print("ANÁLISIS COMPLETADO EXITOSAMENTE")
print("="*60)

RESUMEN DE ANÁLISIS DE VENTAS

1. Datos Cargados:
   - Total de registros: 6
   - Total de columnas: 6

2. Análisis por Categoría:
+-----------+---------------+--------------+----------------+------------------+-------------+-------------+
|Categoria  |Total_Productos|Total_Unidades|Ingresos_Totales|Precio_Promedio   |Precio_Maximo|Precio_Minimo|
+-----------+---------------+--------------+----------------+------------------+-------------+-------------+
|Electrónica|3              |135           |1305000         |10333.333333333334|15000        |7000         |
|Periférico |3              |195           |218000          |1800.0            |4000         |600          |
+-----------+---------------+--------------+----------------+------------------+-------------+-------------+


3. Análisis por Región:
+------+---------------+--------------+----------------+---------------+
|Region|Total_Productos|Total_Unidades|Ingresos_Totales|Precio_Promedio|
+------+---------------+--------------+----

In [ ]:
# Detener sesión de Spark
spark.stop()
print("Sesión de Spark detenida")